# 1. Import Libraries

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path
import pandas as pd
import re
import time
from datetime import datetime

# 2. Define Settings

In [2]:
BASE_URL = 'https://www.fcc.gov/documents?field_edoc_doctype_target_id%5B%5D=69&field_released_date_value=&exposed_from_date=&exposed_to_date='

FCC_HOME = 'https://www.fcc.gov'

PAGE_START = 0
PAGE_END = 1

OUTPUT_FOLDER = Path('fcc_news_release_documents')
METADATA_FILE = OUTPUT_FOLDER / 'fcc_news_release_metadata.csv'

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (academic research)',
    'Accept-Language': 'en',
}

TIMEOUT = 60

# 3. Define Helper Functions

In [3]:
def create_session(headers=HEADERS):
    """
    Create a requests session with browser-like headers.
    """
    session = requests.Session()
    session.headers.update(headers)
    return session


def get_soup(session, url, timeout=TIMEOUT):
    """
    Download a webpage and return a BeautifulSoup object.
    """
    response = session.get(url, timeout=timeout)
    response.raise_for_status()
    return BeautifulSoup(response.text, 'html.parser')


def build_search_url(page):
    """
    Build the FCC search result URL for a given page number.
    """
    return f'{BASE_URL}&page={page}'


def clean_text(text):
    """
    Clean extra spaces from extracted text.
    """
    if text is None:
        return None
    return re.sub(r'\s+', ' ', text).strip()


def clean_filename(title, max_length=150):
    """
    Convert a document title into a safe file name.
    """
    clean_title = re.sub(r'[\\/*?:"<>|]', '', title)
    clean_title = re.sub(r'\s+', '_', clean_title)
    clean_title = clean_title.strip('_')

    if not clean_title:
        clean_title = 'untitled'

    return clean_title[:max_length]


def format_date_for_filename(date_text):
    """
    Convert FCC date text like 'Apr 29, 2026' into '260429'.
    """
    if not date_text:
        return 'unknown_date'

    date_text = date_text.strip()

    for date_format in ['%b %d, %Y', '%B %d, %Y']:
        try:
            date_object = datetime.strptime(date_text, date_format)
            return date_object.strftime('%y%m%d')
        except ValueError:
            continue

    return 'unknown_date'


def get_one_text(soup, selector):
    """
    Extract one text item from a CSS selector.
    """
    tag = soup.select_one(selector)
    if tag is None:
        return None
    return clean_text(tag.get_text(' ', strip=True))


def get_many_texts(soup, selector):
    """
    Extract multiple text items from a CSS selector.
    """
    texts = []

    for tag in soup.select(selector):
        text = clean_text(tag.get_text(' ', strip=True))
        if text:
            texts.append(text)

    return texts


def get_date_text(soup, selector):
    """
    Extract date text from FCC date fields.
    """
    tag = soup.select_one(selector)

    if tag is None:
        return None

    time_tag = tag.select_one('time')

    if time_tag is None:
        return None

    return clean_text(time_tag.get_text(' ', strip=True))


def normalize_label(text):
    """
    Normalize document captions and document types for comparison.
    Example: 'News Release:' -> 'news release'
    """
    text = clean_text(text) or ''
    text = re.sub(r'\s*:\s*$', '', text)
    text = text.lower()
    text = re.sub(r'[^a-z0-9]+', ' ', text)
    return text.strip()


def extract_news_release_txt_urls(soup):
    """
    Extract the main News Release txt file from an FCC document page.
    Ignore supplementary files such as statements or related documents.
    """
    selected_txt_urls = []
    all_txt_urls = []

    for block in soup.select('span.edoc-document'):
        caption_tag = block.select_one('.document-caption')
        txt_tag = block.select_one('a.document-link.txt[href]')

        if txt_tag is None:
            continue

        txt_url = urljoin(FCC_HOME, txt_tag.get('href'))
        all_txt_urls.append(txt_url)

        if caption_tag is None:
            continue

        caption = clean_text(caption_tag.get_text(' ', strip=True))
        caption_norm = normalize_label(caption)

        if caption_norm == 'news release':
            selected_txt_urls.append(txt_url)

    return selected_txt_urls, all_txt_urls


def extract_document_metadata(session, document_url):
    """
    Open one FCC document webpage and extract metadata and txt file URLs.
    """
    soup = get_soup(session, document_url)

    full_title = get_one_text(soup, '.edoc__full-title .field__item')
    page_title = get_one_text(soup, 'h1.display-4 span') or get_one_text(soup, 'h1')

    document_type = get_one_text(soup, '.edoc__doctype .field__item')
    bureaus = get_many_texts(soup, '.edoc__field-bureau-office-category .field__item')

    description = get_one_text(soup, '.edoc__short-desc .field__item')

    released_on = get_date_text(soup, '.edoc__release-dt')
    issued_on = get_date_text(soup, '.edoc__publish-dt')
    adopted = get_date_text(soup, '.edoc__adopted-dt')

    selected_txt_urls, all_txt_urls = extract_news_release_txt_urls(soup)

    metadata = {
        'page_title': page_title,
        'full_title': full_title,
        'document_type': document_type,
        'bureaus': '; '.join(bureaus) if bureaus else None,
        'description': description,
        'webpage_url': document_url,
        'selected_txt_urls': selected_txt_urls,
        'selected_txt_count': len(selected_txt_urls),
        'all_txt_urls': all_txt_urls,
        'all_txt_count': len(all_txt_urls),
        'released_on': released_on,
        'issued_on': issued_on,
        'adopted': adopted,
        'downloaded_files': [],
        'download_status': None,
        'error_message': None
    }

    return metadata


def extract_document_links_from_soup(soup):
    """
    Extract document titles and document page links from one FCC search result page.
    """
    documents = []

    for a in soup.select('div.headline-title a.title'):
        title = a.get_text(strip=True)
        href = a.get('href')

        if not href:
            continue

        link = urljoin(FCC_HOME, href)

        documents.append({
            'webpage_title': title,
            'webpage_url': link
        })

    return documents


def collect_document_links(session, start_page=PAGE_START, end_page=PAGE_END, delay=1):
    """
    Go through FCC search result pages and collect document webpage links.
    """
    document_links = []

    for page in range(start_page, end_page + 1):
        print('=' * 60)
        print(f'Search page {page}')

        page_url = build_search_url(page)

        try:
            soup = get_soup(session, page_url)
            page_documents = extract_document_links_from_soup(soup)

            document_links.extend(page_documents)

            for doc in page_documents:
                print(doc['webpage_title'], '>>>', doc['webpage_url'])

            print(f'Documents found on page {page}: {len(page_documents)}')

        except requests.RequestException as e:
            print(f'>>> Failed to collect page {page}: {e}')

        time.sleep(delay)

    print('\nTotal number of document links:', len(document_links))

    return document_links


def save_txt_file(session, txt_url, file_path, timeout=TIMEOUT):
    """
    Download one txt file and save it.
    """
    response = session.get(txt_url, timeout=timeout)
    response.raise_for_status()
    
    file_path.write_text(response.text, encoding='utf-8')


def download_news_release_documents(session, document_links, output_folder=OUTPUT_FOLDER, delay=1):
    """
    For each FCC News Release page:
    1. collect metadata,
    2. select the txt file matching the document type,
    3. download the selected txt file,
    4. store metadata in a dataframe.
    """
    output_folder.mkdir(exist_ok=True)

    metadata_records = []

    for i, doc in enumerate(document_links, start=1):
        print('=' * 60)
        print(f"Document_{i}: {doc['webpage_title']}")

        try:
            metadata = extract_document_metadata(
                session=session,
                document_url=doc['webpage_url']
            )

            title_for_filename = (
                metadata['full_title']
                or metadata['page_title']
                or doc['webpage_title']
                or f'document_{i}'
            )

            date_for_filename = (
                format_date_for_filename(metadata['released_on'])
                if metadata['released_on']
                else format_date_for_filename(metadata['issued_on'])
            )

            selected_txt_urls = metadata['selected_txt_urls']

            if not selected_txt_urls:
                print('>>> No matching txt file found.')
                metadata['download_status'] = 'skipped_no_matching_txt'
                metadata_records.append(metadata)
                continue

            downloaded_files = []

            for j, txt_url in enumerate(selected_txt_urls, start=1):
                clean_title = clean_filename(title_for_filename)

                if len(selected_txt_urls) == 1:
                    file_name = f'{date_for_filename}_{clean_title}.txt'
                else:
                    file_name = f'{date_for_filename}_{clean_title}_{j}.txt'

                file_path = output_folder / file_name

                save_txt_file(session, txt_url, file_path)

                downloaded_files.append(str(file_path))
                print('Saved:', file_path)

            metadata['downloaded_files'] = downloaded_files
            metadata['download_status'] = 'downloaded'

            metadata_records.append(metadata)

        except requests.RequestException as e:
            print(f'>>> Request failed for document {i}: {e}')

            metadata_records.append({
                'page_title': doc.get('webpage_title'),
                'full_title': None,
                'document_type': None,
                'bureaus': None,
                'description': None,
                'webpage_url': doc.get('webpage_url'),
                'selected_txt_urls': [],
                'selected_txt_count': 0,
                'all_txt_urls': [],
                'all_txt_count': 0,
                'released_on': None,
                'issued_on': None,
                'adopted': None,
                'downloaded_files': [],
                'download_status': 'failed_request',
                'error_message': str(e)
            })

        time.sleep(delay)

    metadata_df = pd.DataFrame(metadata_records)

    # Save a CSV-friendly version.
    metadata_df_for_csv = metadata_df.copy()

    list_columns = [
        'selected_txt_urls',
        'all_txt_urls',
        'downloaded_files'
    ]

    for col in list_columns:
        metadata_df_for_csv[col] = metadata_df_for_csv[col].apply(
            lambda x: ' | '.join(map(str, x)) if isinstance(x, list) else x
        )

    metadata_df_for_csv.to_csv(
        METADATA_FILE,
        index=False,
        encoding='utf-8-sig'
    )

    print('\nDownload summary')
    print('Downloaded:', (metadata_df['download_status'] == 'downloaded').sum())
    print('Skipped:', (metadata_df['download_status'] == 'skipped_no_matching_txt').sum())
    print('Failed:', (metadata_df['download_status'] == 'failed_request').sum())
    print('Metadata saved to:', METADATA_FILE)

    return metadata_df

# 4. Collect Document Links from FCC Search Result Pages

In [4]:
session = create_session()

document_links = collect_document_links(
    session=session,
    start_page=PAGE_START,
    end_page=PAGE_END,
    delay=1
)

Search page 0
FCC Proposes to Amend Audible Crawl Rule to Preserve Accessibility >>> https://www.fcc.gov/document/fcc-proposes-amend-audible-crawl-rule-preserve-accessibility
FCC Adopts Rules to Enhance the Integrity of the E-Rate Program >>> https://www.fcc.gov/document/fcc-adopts-rules-enhance-integrity-e-rate-program
FCC Targets 'Covered List' Entities' Blanket Sec. 214 Authorizations >>> https://www.fcc.gov/document/fcc-targets-covered-list-entities-blanket-sec-214-authorizations
FCC Targets Device Test Labs in Nations Without Reciprocal Agreements >>> https://www.fcc.gov/document/fcc-targets-device-test-labs-nations-without-reciprocal-agreements
FCC Proposes Strengthening 'Know-Your-Customer' Rules >>> https://www.fcc.gov/document/fcc-proposes-strengthening-know-your-customer-rules
FCC Modernizes Satellite Spectrum Sharing Rules to Boost Broadband >>> https://www.fcc.gov/document/fcc-modernizes-satellite-spectrum-sharing-rules-boost-broadband
What They Are Saying: Stakeholders App

# 5. Download News Release txt files and collect metadata

In [5]:
metadata_df = download_news_release_documents(
    session=session,
    document_links=document_links,
    output_folder=OUTPUT_FOLDER,
    delay=1
)

Document_1: FCC Proposes to Amend Audible Crawl Rule to Preserve Accessibility
Saved: fcc_news_release_documents\260430_FCC_Proposes_to_Amend_Audible_Crawl_Rule_to_Preserve_Accessibility.txt
Document_2: FCC Adopts Rules to Enhance the Integrity of the E-Rate Program
Saved: fcc_news_release_documents\260430_FCC_Adopts_Rules_to_Enhance_the_Integrity_of_the_E-Rate_Program.txt
Document_3: FCC Targets 'Covered List' Entities' Blanket Sec. 214 Authorizations
Saved: fcc_news_release_documents\260430_FCC_Proposes_Banning_Entities_on_'Covered_List'_from_Receiving_Blanket_Authority_to_Provide_Telecommunications_Services.txt
Document_4: FCC Targets Device Test Labs in Nations Without Reciprocal Agreements
Saved: fcc_news_release_documents\260430_FCC_Looks_to_Prohibit_Electronic_Device_Testing_Using_Labs_in_Countries_Without_Reciprocal_Agreements.txt
Document_5: FCC Proposes Strengthening 'Know-Your-Customer' Rules
Saved: fcc_news_release_documents\260430_FCC_Proposes_Strengthening_'Know-Your-Cust

In [6]:
metadata_df.head()

,page_title,full_title,document_type,bureaus,description,webpage_url,selected_txt_urls,selected_txt_count,all_txt_urls,all_txt_count,released_on,issued_on,adopted,downloaded_files,download_status,error_message
0,FCC Proposes to Amend Audible Crawl Rule to Pr...,FCC Proposes to Amend Audible Crawl Rule to Pr...,News Release,Media Relations; Space,None,https://www.fcc.gov/document/fcc-proposes-amen...,[https://docs.fcc.gov/public/attachments/DOC-4...,1,[https://docs.fcc.gov/public/attachments/DOC-4...,3,"Apr 30, 2026","Apr 30, 2026","Apr 30, 2026",[fcc_news_release_documents\260430_FCC_Propose...,downloaded,None
1,FCC Adopts Rules to Enhance the Integrity of t...,FCC Adopts Rules to Enhance the Integrity of t...,News Release,Media Relations; Wireline Competition,None,https://www.fcc.gov/document/fcc-adopts-rules-...,[https://docs.fcc.gov/public/attachments/DOC-4...,1,[https://docs.fcc.gov/public/attachments/DOC-4...,4,"Apr 30, 2026","Apr 30, 2026","Apr 30, 2026",[fcc_news_release_documents\260430_FCC_Adopts_...,downloaded,None
2,FCC Targets 'Covered List' Entities' Blanket S...,FCC Proposes Banning Entities on 'Covered List...,News Release,Media Relations; Wireline Competition,None,https://www.fcc.gov/document/fcc-targets-cover...,[https://docs.fcc.gov/public/attachments/DOC-4...,1,[https://docs.fcc.gov/public/attachments/DOC-4...,3,"Apr 30, 2026","Apr 30, 2026","Apr 30, 2026",[fcc_news_release_documents\260430_FCC_Propose...,downloaded,None
3,FCC Targets Device Test Labs in Nations Withou...,FCC Looks to Prohibit Electronic Device Testin...,News Release,Engineering & Technology; Media Relations,None,https://www.fcc.gov/document/fcc-targets-devic...,[https://docs.fcc.gov/public/attachments/DOC-4...,1,[https://docs.fcc.gov/public/attachments/DOC-4...,3,"Apr 30, 2026","Apr 30, 2026","Apr 30, 2026",[fcc_news_release_documents\260430_FCC_Looks_t...,downloaded,None
4,FCC Proposes Strengthening 'Know-Your-Customer...,FCC Proposes Strengthening 'Know-Your-Customer...,News Release,Consumer and Governmental Affairs; Media Relat...,None,https://www.fcc.gov/document/fcc-proposes-stre...,[https://docs.fcc.gov/public/attachments/DOC-4...,1,[https://docs.fcc.gov/public/attachments/DOC-4...,3,"Apr 30, 2026","Apr 30, 2026","Apr 30, 2026",[fcc_news_release_documents\260430_FCC_Propose...,downloaded,None
